In [1]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_DIR = Path("data/raw")          # yahan BTS ki CSV files rakhi hain
OUT_DIR = Path("outputs")            # charts aur CSV yahan save honge
OUT_DIR.mkdir(exist_ok=True)

In [2]:


# 1. Aapka exact file path (raw string ke sath)
file_path = r"C:\Users\HP\Downloads\On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_1\On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_1.csv"

# 2. Zaroori 17 columns ki list (jo aage ke steps mein kaam aayenge)
selected_columns = [
    'FlightDate', 'Reporting_Airline', 'Origin', 'Dest', 
    'DepDelay', 'ArrDelay', 'Cancelled', 'Diverted', 
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 
    'LateAircraftDelay', 'DepTime', 'ArrTime', 'Distance', 'CancellationCode'
]

# 3. Sirf 17 columns ko load karna
df = pd.read_csv(file_path, usecols=selected_columns, low_memory=False)

# 4. Check karne ke liye ki data successfully load hua ya nahi
print("Data successfully loaded!")
print("Total Rows & Selected Columns:", df.shape)

# Pehle 3 rows dekhein
df.head(3)

Data successfully loaded!
Total Rows & Selected Columns: (544003, 17)


,FlightDate,Reporting_Airline,Origin,Dest,DepTime,DepDelay,ArrTime,ArrDelay,Cancelled,CancellationCode,Diverted,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2026-01-01,AA,JFK,LAX,657.0,-3.0,1104.0,43.0,0.0,NaN,0.0,2475.0,0.0,0.0,43.0,0.0,0.0
1,2026-01-02,AA,JFK,LAX,654.0,-6.0,1022.0,1.0,0.0,NaN,0.0,2475.0,NaN,NaN,NaN,NaN,NaN
2,2026-01-03,AA,JFK,LAX,658.0,-2.0,1011.0,-10.0,0.0,NaN,0.0,2475.0,NaN,NaN,NaN,NaN,NaN


In [3]:
# Step 3: Data Quality Check

print("--- 1. Missing Values (Null check) ---")
print(df.isnull().sum())

print("\n--- 2. Duplicate Rows Check ---")
duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows: {duplicate_count}")

print("\n--- 3. Cancelled & Diverted Flights Count ---")
print("Cancelled:\n", df['Cancelled'].value_counts())
print("Diverted:\n", df['Diverted'].value_counts())

--- 1. Missing Values (Null check) ---
FlightDate                0
Reporting_Airline         0
Origin                    0
Dest                      0
DepTime               25164
DepDelay              25229
ArrTime               25838
ArrDelay              26781
Cancelled                 0
CancellationCode     518368
Diverted                  0
Distance                  0
CarrierDelay         436528
WeatherDelay         436528
NASDelay             436528
SecurityDelay        436528
LateAircraftDelay    436528
dtype: int64

--- 2. Duplicate Rows Check ---
Total duplicate rows: 10084

--- 3. Cancelled & Diverted Flights Count ---
Cancelled:
 Cancelled
0.0    518368
1.0     25635
Name: count, dtype: int64
Diverted:
 Diverted
0.0    542857
1.0      1146
Name: count, dtype: int64


In [4]:
# Step 4: Data Cleaning

# 1. Delay se jude columns ki list
delay_columns = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']

# 2. In columns ki sabhi NaN (khali) values ko 0 se replace karna
df[delay_columns] = df[delay_columns].fillna(0)

# 3. Check karna ki ab koi null value bachi hai ya nahi
print("--- Cleaning ke baad Delay columns ke Null values count ---")
print(df[delay_columns].isnull().sum())

# 4. Data ka ek chota view dekhne ke liye
print("\nCleaned Data Sample:")
df[delay_columns].head(3)

--- Cleaning ke baad Delay columns ke Null values count ---
CarrierDelay         0
WeatherDelay         0
NASDelay             0
SecurityDelay        0
LateAircraftDelay    0
dtype: int64

Cleaned Data Sample:


,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,0.0,0.0,43.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0


In [5]:
# Step 5: Feature Engineering (Naye columns banana)

import numpy as np

# 1. is_delayed column banana (Agar ArrDelay > 0 hai toh 1, warna 0)
df['is_delayed'] = np.where(df['ArrDelay'] > 0, 1, 0)

# 2. Route column banana (Origin + Destination)
df['Route'] = df['Origin'] + ' -> ' + df['Dest']

# 3. DepHour banana (DepTime ko 100 se divide karke hour nikalna, NaN ko handle karte hue)
# Note: DepTime jaise '650' matlab 6th hour hai
df['DepHour'] = pd.to_numeric(df['DepTime'], errors='coerce') // 100
df['DepHour'] = df['DepHour'].fillna(0).astype(int)

# Check karne ke liye naye columns ki pehle 3 rows dekhna
print("Naye columns ke sath Data Sample:")
print(df[['FlightDate', 'Reporting_Airline', 'Route', 'DepDelay', 'ArrDelay', 'is_delayed', 'DepHour']].head(3))

Naye columns ke sath Data Sample:
   FlightDate Reporting_Airline       Route  DepDelay  ArrDelay  is_delayed  DepHour
0  2026-01-01                AA  JFK -> LAX      -3.0      43.0           1        6
1  2026-01-02                AA  JFK -> LAX      -6.0       1.0           1        6
2  2026-01-03                AA  JFK -> LAX      -2.0     -10.0           0        6


In [6]:
# Step 6: SQL Database mein data load karna

import sqlite3

# 1. SQLite database connection banana (Yeh aapke folder mein 'flights.db' naam ki file bana dega)
conn = sqlite3.connect('flights.db')

# 2. DataFrame ko SQL table ('flight_data') mein convert karna
df.to_sql('flight_data', conn, if_exists='replace', index=False)

print("Data successfully SQL database ('flights.db') mein table 'flight_data' ke roop mein save ho chuka hai!")

Data successfully SQL database ('flights.db') mein table 'flight_data' ke roop mein save ho chuka hai!


In [8]:
# Step 7: Monthly-wise trend with SQL LAG function


# 1. Database se connection dobara banana
conn = sqlite3.connect('flights.db')

# 2. SQL Query likhna jismein mahine ke hisaab se delays count hon aur LAG lagega
query = """
WITH MonthlyStats AS (
    SELECT 
        substr(FlightDate, 1, 7) AS Month,
        COUNT(*) AS Total_Flights,
        SUM(is_delayed) AS Total_Delays
    FROM flight_data
    GROUP BY Month
)
SELECT 
    Month,
    Total_Flights,
    Total_Delays,
    LAG(Total_Delays, 1) OVER (ORDER BY Month) AS Prev_Month_Delays,
    Total_Delays - LAG(Total_Delays, 1) OVER (ORDER BY Month) AS Delay_Difference
FROM MonthlyStats;
"""

# 3. Query ko execute karke DataFrame mein laana
df_trend = pd.read_sql(query, conn)

# 4. Output dekhna
print("Monthly-wise Delay Trend (with LAG):")
print(df_trend)

Monthly-wise Delay Trend (with LAG):
     Month  Total_Flights  Total_Delays Prev_Month_Delays Delay_Difference
0  2026-01         544003        180735              None             None


In [9]:
# Step 8: Airline Ranking using SQL RANK() function

# 2. SQL Query likhna jismein RANK() function ka use ho
query = """
WITH AirlineStats AS (
    SELECT 
        Reporting_Airline AS Airline,
        COUNT(*) AS Total_Flights,
        SUM(is_delayed) AS Total_Delays,
        ROUND(AVG(ArrDelay), 2) AS Avg_Arr_Delay
    FROM flight_data
    GROUP BY Reporting_Airline
)
SELECT 
    Airline,
    Total_Flights,
    Total_Delays,
    Avg_Arr_Delay,
    RANK() OVER (ORDER BY Total_Delays DESC) AS Delay_Rank
FROM AirlineStats;
"""

# 3. Query ko execute karke DataFrame mein laana
df_airline_rank = pd.read_sql(query, conn)

# 4. Output dekhna
print("Airline Ranking based on Total Delays:")
print(df_airline_rank)

Airline Ranking based on Total Delays:
   Airline  Total_Flights  Total_Delays  Avg_Arr_Delay  Delay_Rank
0       WN         104002         31424          -1.22           1
1       AA          76772         26841          13.81           2
2       DL          79198         25534           3.93           3
3       OO          66168         22710          13.17           4
4       UA          63781         19839           2.02           5
5       AS          26436         10238           4.34           6
6       YX          29572          8801           2.51           7
7       B6          18109          7787          16.09           8
8       MQ          24155          7745           4.98           9
9       OH          19550          6156          14.03          10
10      F9          15382          6017          13.57          11
11      NK          11915          4497          10.40          12
12      G4           8963          3146          10.17          13


In [10]:
# Step 9: Airport-wise delay analysis (Top 10 Airports with most delays)

query = """
SELECT 
    Origin AS Airport,
    COUNT(*) AS Total_Flights,
    SUM(is_delayed) AS Total_Delays,
    ROUND(AVG(DepDelay), 2) AS Avg_Dep_Delay
FROM flight_data
GROUP BY Origin
ORDER BY Total_Delays DESC
LIMIT 10;
"""

# Query run karke dataframe mein laana
df_airport = pd.read_sql(query, conn)

# Output dekhna
print("Top 10 Airports with Highest Delays:")
print(df_airport)

Top 10 Airports with Highest Delays:
  Airport  Total_Flights  Total_Delays  Avg_Dep_Delay
0     ORD          25131         11167          22.55
1     ATL          24595          8820          10.71
2     DFW          24229          8810          18.83
3     DEN          24484          7655           9.71
4     PHX          16564          5558          10.76
5     MCO          13786          5434          14.68
6     LAX          14815          4901          12.18
7     DTW           9518          4545          24.27
8     LAS          13695          4314          10.27
9     DCA          11858          4161          16.64


In [11]:
# Step 10: Routes Pareto Analysis (Cumulative Delay Minutes)

query = """
WITH RouteDelays AS (
    SELECT 
        Route,
        SUM(ArrDelay) AS Total_Delay_Minutes,
        COUNT(*) AS Total_Flights
    FROM flight_data
    WHERE ArrDelay > 0
    GROUP BY Route
),
RankedRoutes AS (
    SELECT 
        Route,
        Total_Delay_Minutes,
        SUM(Total_Delay_Minutes) OVER (ORDER BY Total_Delay_Minutes DESC) AS Cumulative_Delay,
        SUM(Total_Delay_Minutes) OVER () AS Grand_Total_Delay
    FROM RouteDelays
)
SELECT 
    Route,
    Total_Delay_Minutes,
    ROUND((Cumulative_Delay * 100.0) / Grand_Total_Delay, 2) AS Cumulative_Percentage
FROM RankedRoutes
ORDER BY Total_Delay_Minutes DESC
LIMIT 15;
"""

# Query run karke dataframe mein laana
df_pareto = pd.read_sql(query, conn)

# Output dekhna
print("Routes Pareto Analysis (Top routes contributing to delays):")
print(df_pareto)

Routes Pareto Analysis (Top routes contributing to delays):
         Route  Total_Delay_Minutes  Cumulative_Percentage
0   LAX -> JFK              21940.0                   0.25
1   ORD -> LGA              17929.0                   0.46
2   DTW -> ORD              16475.0                   0.65
3   JFK -> LAX              16211.0                   0.84
4   ORD -> MKE              15805.0                   1.03
5   DCA -> BOS              15352.0                   1.20
6   ORD -> DTW              14720.0                   1.38
7   ORD -> DCA              14523.0                   1.54
8   LGA -> ORD              14474.0                   1.71
9   BOS -> DCA              14354.0                   1.88
10  DFW -> ORD              13677.0                   2.04
11  ATL -> FLL              13240.0                   2.19
12  MSN -> ORD              13010.0                   2.34
13  ORD -> MSP              12843.0                   2.49
14  ATL -> MIA              12809.0                   2

In [12]:
# Step 11: Delay Causes Analysis (Total minutes by each delay category)

query = """
SELECT 
    SUM(CarrierDelay) AS Carrier_Delay,
    SUM(WeatherDelay) AS Weather_Delay,
    SUM(NASDelay) AS NAS_Delay,
    SUM(SecurityDelay) AS Security_Delay,
    SUM(LateAircraftDelay) AS Late_Aircraft_Delay
FROM flight_data;
"""

# Query run karke dataframe mein laana
df_delay_causes = pd.read_sql(query, conn)

# Data ko achhi readability ke liye melt (transpose) karna aur descending order mein lagana
df_causes_melted = df_delay_causes.melt(var_name='Delay_Type', value_name='Total_Minutes')
df_causes_melted = df_causes_melted.sort_values(by='Total_Minutes', ascending=False).reset_index(drop=True)

print("Total Delay Minutes by Root Causes:")
print(df_causes_melted)

Total Delay Minutes by Root Causes:
            Delay_Type  Total_Minutes
0        Carrier_Delay      3031178.0
1  Late_Aircraft_Delay      2842019.0
2            NAS_Delay      1465808.0
3        Weather_Delay       776908.0
4       Security_Delay        11479.0


In [13]:
# Step 12: Hourly Delay Analysis (Delays by Departure Hour)

query = """
SELECT 
    DepHour AS Departure_Hour,
    COUNT(*) AS Total_Flights,
    SUM(is_delayed) AS Total_Delays,
    ROUND(AVG(DepDelay), 2) AS Avg_Dep_Delay
FROM flight_data
GROUP BY DepHour
ORDER BY DepHour ASC;
"""

# Query run karke dataframe mein laana
df_hourly = pd.read_sql(query, conn)

# Output dekhna
print("Delays by Hour of the Day:")
print(df_hourly)

Delays by Hour of the Day:
    Departure_Hour  Total_Flights  Total_Delays  Avg_Dep_Delay
0                0          26999          1140          83.54
1                1            564           417         132.09
2                2            236           152         117.34
3                3             66            42         127.74
4                4           1943           232          -3.55
5                5          20530          3440          -3.87
6                6          31218          7178          -0.90
7                7          32638          8875           2.39
8                8          35213          9851           3.98
9                9          29003          9045           9.05
10              10          31734          9992          10.68
11              11          32419         10867          11.14
12              12          32391         10817          12.61
13              13          31439         11076          14.28
14              14          

In [14]:
# Step 13: Cancellation Reasons Breakdown

query = """
SELECT 
    CASE 
        WHEN CancellationCode = 'A' THEN 'Carrier'
        WHEN CancellationCode = 'B' THEN 'Weather'
        WHEN CancellationCode = 'C' THEN 'NAS (National Air System)'
        WHEN CancellationCode = 'D' THEN 'Security'
        ELSE 'Not Specified / Other'
    END AS Cancellation_Reason,
    COUNT(*) AS Total_Cancelled_Flights
FROM flight_data
WHERE Cancelled = 1
GROUP BY CancellationCode
ORDER BY Total_Cancelled_Flights DESC;
"""

# Query run karke dataframe mein laana
df_cancellation = pd.read_sql(query, conn)

# Output dekhna
print("Cancellation Reasons Breakdown:")
print(df_cancellation)

Cancellation Reasons Breakdown:
         Cancellation_Reason  Total_Cancelled_Flights
0                    Weather                    22463
1                    Carrier                     2246
2  NAS (National Air System)                      879
3                   Security                       47


In [15]:
# Step 14: Distance Category ke hisaab se Delay Analysis

query = """
SELECT 
    CASE 
        WHEN Distance < 500 THEN '1. Short Haul (< 500 miles)'
        WHEN Distance BETWEEN 500 AND 1500 THEN '2. Medium Haul (500-1500 miles)'
        ELSE '3. Long Haul (> 1500 miles)'
    END AS Distance_Category,
    COUNT(*) AS Total_Flights,
    SUM(is_delayed) AS Total_Delays,
    ROUND(AVG(ArrDelay), 2) AS Avg_Arr_Delay
FROM flight_data
GROUP BY Distance_Category
ORDER BY Distance_Category ASC;
"""

# Query run karke dataframe mein laana
df_distance = pd.read_sql(query, conn)

# Output dekhna
print("Delay Analysis by Flight Distance Categories:")
print(df_distance)

Delay Analysis by Flight Distance Categories:
                 Distance_Category  Total_Flights  Total_Delays  Avg_Arr_Delay
0      1. Short Haul (< 500 miles)         181156         57330           6.88
1  2. Medium Haul (500-1500 miles)         291288         97593           6.31
2      3. Long Haul (> 1500 miles)          71559         25812           5.62


In [16]:
# Step 15: Close Database Connection

# Close the database connection safely
conn.close()

print("Database connection successfully closed!")
print("Congratulations! Your 15-step End-to-End Flight Data Analysis project has been successfully completed!")

Database connection successfully closed!
Congratulations! Your 15-step End-to-End Flight Data Analysis project has been successfully completed!


In [17]:
# Cleaned dataframe ko CSV mein save karna
df.to_csv('cleaned_flight_data.csv', index=False)
print("Cleaned CSV file saved successfully!")

Cleaned CSV file saved successfully!
